# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [ ]:
df['revenue'] = df.qty * df.price

print(len(df))
print(df.revenue.sum())
print(df.qty.sum())

400
8520.0
783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [ ]:
cat = df.groupby('category').revenue.sum()
table = cat.sort_values(ascending=False).reset_index()

total = df['revenue'].sum()
table['percent'] = (table['revenue'] / total * 100).round(1)
table



,category,revenue,percent
0,Food,4293.0,50.4
1,Merch,1771.5,20.8
2,Drink,1554.0,18.2
3,RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [ ]:
vend = df.groupby('vendor_id')['revenue']

table = vend.count().to_frame('orders')
table['avg_revenue'] = vend.mean()
table = table.sort_values('avg_revenue', ascending=False)

table

,orders,avg_revenue
vendor_id,,
V-01,94,22.595745
V-18,108,21.750000
V-05,93,20.580645
V-10,105,20.314286


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [ ]:
merch_rev = df[df['category']=='Merch']['revenue'].sum()
total_rev = df['revenue'].sum()

print ((merch_rev / total_rev * 100).round(2))

20.79


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [ ]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

merged = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

print(len(df))
print(df['revenue'].sum())
print(len(merged))
print(merged['revenue'].sum())

400
8520.0
400
8520.0


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [ ]:
pivot = pd.pivot_table(merged, values='revenue', index='vendor_name', columns='category', aggfunc='sum', margins=True, margins_name='Total')
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Total,972.0,3274.5,1263.0,661.5,6171.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [ ]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()

cat = df.groupby('category').revenue.sum()
by_category = cat.sort_values(ascending=False).reset_index()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.



*   I would tell the vendors to maybe reconsider RainGear, as it made up the least amount of revenue (10.6%). But since we don't have more information about the circumstances of the day, this isn't an entirely backed recommendation. I would also recommend to keep food as a main focus because it made up half of revenue (50.4%) across the board, and also about half for each individual vendor.
*   Q3, because tha average is not uniform across the vendors. Meaning that they are on different order numbers. Even though in this data, the numbers aren't more than about 10 apart at most, the mmetric itself is just not that useful.


